# R7 - Registered agent (2 versions, prompt SHA tags, champion alias) + serving endpoint

GenAi-2 (Paper 5) R7: turn the R2-pattern agent into a governed, servable product.

Deliverables:
1. Registered model `claims.genai.claims_agent` with TWO versions: v1 <- `system_v1.md`, v2 <- `system_v2.md`; tag `system_prompt_sha` on each version = first 12 hex chars of SHA-256 of the FULL prompt file text (leading/trailing whitespace removed); alias `champion` on the v2 version.
2. `agent/prompts/system_v2.md`: >= 40 words, different from `system_v1.md` (written in R4; validated here, not rewritten).
3. CPU serving endpoint `claims-agent` serving the champion version; screenshot: `evidence/screenshots/r7_endpoint.png`.
4. `evidence/r7_registry.json`: key `mode` ("uc"; "experiment" is the permitted fallback) + versions/tags/alias/endpoint record.

In [0]:
# 0. Install (same pinned set as R6; NO kernel restart - R5 lesson: a restart kills the protobuf>=5 boot).
%pip install --quiet "langchain-databricks==0.1.2" "langchain-core==0.3.86" "mlflow-skinny<3" "mcp==1.16.0" "databricks-mcp==0.1.0"
%pip install --quiet "protobuf>=6.31,<7"

In [0]:
# 1. Config + the shared MLflow experiment (R2-R6 traces live here too).
import hashlib, json, time

CATALOG, SCHEMA = "claims", "genai"
MODEL_NAME = f"{CATALOG}.{SCHEMA}.claims_agent"
ENDPOINT_NAME = "claims-agent"
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"
TEMPERATURE = 0.0
WAREHOUSE_ID = "78d419cb76b1d7e9"  # Serverless Starter Warehouse (R1-R6)
GENAI2_ROOT = "/Workspace/Users/custom001x@gmail.com/saffron-shield/GenAi-2"
V1_PATH = GENAI2_ROOT + "/agent/prompts/system_v1.md"
V2_PATH = GENAI2_ROOT + "/agent/prompts/system_v2.md"
EVIDENCE_PATH = GENAI2_ROOT + "/evidence/r7_registry.json"
SCREENSHOT_REL = "evidence/screenshots/r7_endpoint.png"
EXPERIMENT = "/Users/custom001x@gmail.com/saffron-shield/genai2-agent"

import mlflow
mlflow.set_tracking_uri("databricks")
mlflow.set_experiment(EXPERIMENT)
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
print(f"model={MODEL_NAME} | endpoint={ENDPOINT_NAME} | chat={CHAT_MODEL}")

In [0]:
# 2. The two prompts and their SHA-256 fingerprints.
#    v1 = the R2 file, untouched. v2 = agent/prompts/system_v2.md (written in R4 = v1 + memory rule):
#    it already satisfies R7 (>= 40 words, different from v1), so it is validated here, NOT rewritten.
def prompt_sha(path):
    text = open(path).read()
    return hashlib.sha256(text.strip().encode()).hexdigest()[:12], text.strip()

V1_SHA, V1_TEXT = prompt_sha(V1_PATH)
V2_SHA, V2_TEXT = prompt_sha(V2_PATH)
assert V1_TEXT != V2_TEXT, "v2 must differ from v1"
assert len(V2_TEXT.split()) >= 40, "v2 needs >= 40 words"
print(f"v1 sha={V1_SHA} ({len(V1_TEXT.split())} words)")
print(f"v2 sha={V2_SHA} ({len(V2_TEXT.split())} words)")

In [0]:
# 3. The agent as an mlflow pyfunc model, logged TWICE: v1 <- system_v1.md, v2 <- system_v2.md.
#    Registered as claims.genai.claims_agent; tag system_prompt_sha on each version; alias champion on v2;
#    the MLflow runs carry agent_version 1/2 tags (the v2 run also carries alias=champion).
import mlflow.pyfunc
import shutil, tempfile
from mlflow.models import ModelSignature
from mlflow.types import Schema, ColSpec
from mlflow.tracking import MlflowClient
client = MlflowClient()

class ClaimsAgent(mlflow.pyfunc.PythonModel):
    """Single-turn R2-pattern agent (ChatDatabricks + tool loop). The system prompt is loaded
    from the model artifact, so v1/v2 differ ONLY in that file - exactly what the SHA tags prove."""

    def load_context(self, context):
        self.system_prompt = open(context.artifacts["system_prompt"]).read()
        self._w = None

    @property
    def w(self):
        if self._w is None:
            self._w = WorkspaceClient()  # notebook auth here; the endpoint SP's auth when served
        return self._w

    def _sql_json(self, sql):
        from databricks.sdk.service.sql import Disposition, Format, StatementState
        resp = self.w.statement_execution.execute_statement(
            warehouse_id=WAREHOUSE_ID, statement=sql,
            disposition=Disposition.INLINE, format=Format.JSON_ARRAY, wait_timeout="50s")
        while resp.status.state in (StatementState.PENDING, StatementState.RUNNING):
            time.sleep(2)
            resp = self.w.statement_execution.get_statement(resp.statement_id)
        if resp.status.state != StatementState.SUCCEEDED:
            msg = resp.status.error.message if resp.status.error else str(resp.status.state)
            raise RuntimeError(f"SQL failed: {msg}")
        rows = resp.result.data_array or []
        return [json.loads(r[0]) for r in rows if r and r[0]]

    # --- the four R1 tools (thin SQL wrappers over the UC functions) ---
    def tool_lookup_claim(self, claim_id):
        rows = self._sql_json(f"SELECT to_json(struct(*)) FROM claims.genai.lookup_claim('{claim_id}') LIMIT 1")
        return rows[0] if rows else None

    def tool_plan_at(self, member_id, as_of_date):
        rows = self._sql_json(f"SELECT to_json(struct(*)) FROM claims.genai.plan_at('{member_id}', '{as_of_date}') LIMIT 1")
        return rows[0] if rows else None

    def tool_search_policy(self, query):
        rows = self._sql_json("SELECT to_json(named_struct('chunks', collect_list(named_struct('chunk_id', c.chunk_id, 'section_title', c.section_title, 'chunk_text', c.chunk_text)))) FROM (SELECT * FROM claims.genai.search_policy('" + query.replace("'", " ") + "') LIMIT 5) c")
        return rows[0] if rows else {"chunks": []}

    def tool_compute_payable(self, plan_code, is_pre_existing, policy_tenure_months, days_since_policy_start, is_network_at_admission, room_rent_per_day, billed_amount):
        sql = (f"SELECT claims.genai.compute_payable('{plan_code}', {str(bool(is_pre_existing)).lower()}, "
               f"{int(policy_tenure_months)}, {int(days_since_policy_start)}, {str(bool(is_network_at_admission)).lower()}, "
               f"{float(room_rent_per_day)}, {float(billed_amount)})")
        rows = self._sql_json(sql)
        return rows[0] if rows else None

    def predict(self, context, model_input):
        from langchain_databricks import ChatDatabricks
        from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

        if hasattr(model_input, "iloc"):
            question = model_input.iloc[0]["question"]
        elif isinstance(model_input, dict):
            question = (model_input.get("question")
                        or (model_input.get("dataframe_records") or [{}])[0].get("question")
                        or (model_input.get("inputs") or {}).get("question"))
        else:
            question = model_input[0]["question"]

        # Plain OpenAI function specs (fully picklable - no local pydantic classes, serving-safe).
        specs = [
            {"type": "function", "function": {"name": "lookup_claim", "description": "All stored facts for one claim",
             "parameters": {"type": "object", "properties": {"claim_id": {"type": "string"}}, "required": ["claim_id"]}}},
            {"type": "function", "function": {"name": "plan_at", "description": "Plan terms for a member on a date",
             "parameters": {"type": "object", "properties": {"member_id": {"type": "string"}, "as_of_date": {"type": "string"}}, "required": ["member_id", "as_of_date"]}}},
            {"type": "function", "function": {"name": "search_policy", "description": "Hybrid keyword+meaning search over policy documents",
             "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
            {"type": "function", "function": {"name": "compute_payable", "description": "Deterministic payout rules engine",
             "parameters": {"type": "object", "properties": {"plan_code": {"type": "string"}, "is_pre_existing": {"type": "boolean"}, "policy_tenure_months": {"type": "integer"}, "days_since_policy_start": {"type": "integer"}, "is_network_at_admission": {"type": "boolean"}, "room_rent_per_day": {"type": "number"}, "billed_amount": {"type": "number"}}, "required": ["plan_code", "is_pre_existing", "policy_tenure_months", "days_since_policy_start", "is_network_at_admission", "room_rent_per_day", "billed_amount"]}}},
        ]
        fns = {"lookup_claim": self.tool_lookup_claim, "plan_at": self.tool_plan_at,
               "search_policy": self.tool_search_policy, "compute_payable": self.tool_compute_payable}
        llm = ChatDatabricks(endpoint=CHAT_MODEL, temperature=TEMPERATURE).bind_tools(specs)
        messages = [SystemMessage(content=self.system_prompt), HumanMessage(content=question)]
        cited, used = [], []
        for _ in range(6):
            ai = llm.invoke(messages)
            messages.append(ai)
            if not ai.tool_calls:
                return {"answer": ai.content, "tool_calls": used, "cited_chunk_ids": cited}
            for call in ai.tool_calls:
                fn = fns[call["name"]]
                result = fn(**call["args"])
                used.append(call["name"])
                if call["name"] == "search_policy":
                    for ch in (result or {}).get("chunks", []):
                        if ch.get("chunk_id") and ch["chunk_id"] not in cited:
                            cited.append(ch["chunk_id"])
                messages.append(ToolMessage(content=json.dumps(result), tool_call_id=call["id"], name=call["name"]))
        return {"answer": "Tool loop did not converge.", "tool_calls": used, "cited_chunk_ids": cited}

SIG = ModelSignature(inputs=Schema([ColSpec("string", "question")]),
                     outputs=Schema([ColSpec("string", "answer_json")]))

# Declare the Databricks resources the agent needs -> serving grants the endpoint's service
# principal automatic authentication passthrough to exactly these (docs: log-agent#specify-resources).
from mlflow.models.resources import DatabricksFunction, DatabricksSQLWarehouse, DatabricksServingEndpoint
RESOURCES = [
    DatabricksServingEndpoint(endpoint_name=CHAT_MODEL),
    DatabricksSQLWarehouse(warehouse_id=WAREHOUSE_ID),
    DatabricksFunction(function_name=f"{CATALOG}.{SCHEMA}.lookup_claim"),
    DatabricksFunction(function_name=f"{CATALOG}.{SCHEMA}.plan_at"),
    DatabricksFunction(function_name=f"{CATALOG}.{SCHEMA}.search_policy"),
    DatabricksFunction(function_name=f"{CATALOG}.{SCHEMA}.compute_payable"),
]

def log_and_register(prompt_path, sha, agent_version, alias=None):
    tmp = tempfile.mkdtemp()
    art = tmp + "/system_prompt.md"
    shutil.copyfile(prompt_path, art)
    with mlflow.start_run(run_name=f"r7_agent_v{agent_version}") as run:
        mlflow.set_tags({"agent_version": str(agent_version), "system_prompt_sha": sha, "round": "r7"})
        mlflow.pyfunc.log_model(
            artifact_path="agent", python_model=ClaimsAgent(),
            artifacts={"system_prompt": art},
            pip_requirements=["databricks-sdk>=0.49.0", "pydantic<3", "langchain-core==0.3.86", "langchain-databricks==0.1.2", "mlflow>=2.16.0,<3"],
            signature=SIG, input_example={"question": "What is the status of claim CLM-10642?"},
            resources=RESOURCES,
            registered_model_name=MODEL_NAME)
        if alias:
            mlflow.set_tag("alias", alias)
    vers = [int(mv.version) for mv in client.search_model_versions(f"name='{MODEL_NAME}'")]
    ver = str(max(vers))
    client.set_model_version_tag(MODEL_NAME, ver, "system_prompt_sha", sha)
    if alias:
        client.set_registered_model_alias(MODEL_NAME, alias, ver)
    return run.info.run_id, ver

V1_RUN, V1_VER = log_and_register(V1_PATH, V1_SHA, 1)
V2_RUN, V2_VER = log_and_register(V2_PATH, V2_SHA, 2, alias="champion")
print(f"v1: run={V1_RUN} version={V1_VER} sha={V1_SHA}")
print(f"v2: run={V2_RUN} version={V2_VER} sha={V2_SHA} alias=champion")

In [0]:
# 4. CPU serving endpoint claims-agent serving the CHAMPION version (v2).
from databricks.sdk.service.serving import ServedEntityInput, EndpointCoreConfigInput
try:
    from databricks.sdk.service.serving import WorkloadType
    WORKLOAD_CPU = WorkloadType.CPU
except ImportError:  # older SDKs: provide an enum-like shim (as_dict() calls .value)
    class _WTV:
        value = "CPU"
    WORKLOAD_CPU = _WTV()
from databricks.sdk.errors.platform import NotFound

def get_endpoint():
    try:
        return w.serving_endpoints.get(ENDPOINT_NAME)
    except NotFound:
        return None

def ready_state(ep):
    try:
        return ep.state.ready.value if ep and ep.state and ep.state.ready else None
    except Exception:
        return None

ep = get_endpoint()
if ep is None:
    w.serving_endpoints.create(name=ENDPOINT_NAME, config=EndpointCoreConfigInput(
        name=ENDPOINT_NAME,
        served_entities=[ServedEntityInput(entity_name=MODEL_NAME, entity_version=str(V2_VER),
                                      workload_type=WORKLOAD_CPU, workload_size="Small",
                                      scale_to_zero_enabled=True)]))
    print("endpoint creation requested")
else:
    served = ep.config.served_entities[0]
    if str(served.entity_version) != str(V2_VER):
        w.serving_endpoints.update_config(
            name=ENDPOINT_NAME,
            served_entities=[ServedEntityInput(entity_name=MODEL_NAME, entity_version=str(V2_VER),
                                               workload_type=WORKLOAD_CPU, workload_size="Small",
                                               scale_to_zero_enabled=True)])
        print(f"endpoint updated: v{served.entity_version} -> v{V2_VER}")
    else:
        print(f"endpoint already serving v{V2_VER}")

# wait (up to ~10 min) until the TARGET version is actually deployed: the endpoint can report
# READY at the top level while a config update (old -> new version) is still IN_PROGRESS.
def deployed_version(e):
    try:
        return str(e.config.served_entities[0].entity_version)
    except Exception:
        return None

def config_busy(e):
    try:
        return (e.state.config_update or "") == "IN_PROGRESS"
    except Exception:
        return False

deadline = time.time() + 600
while time.time() < deadline:
    ep = get_endpoint()
    dv, busy = deployed_version(ep), config_busy(ep)
    print(f"served v{dv} | config_update={'IN_PROGRESS' if busy else 'settled'}", flush=True)
    if dv == str(V2_VER) and not busy and ready_state(ep) == "READY":
        print(f"v{V2_VER} is LIVE")
        break
    time.sleep(30)
else:
    print(f"NOTE: v{V2_VER} not deployed yet - check the endpoint UI (Deployments tab) and re-run this cell")

In [0]:
# 5. evidence/r7_registry.json - the receipt (mode "uc" = real UC registry; "experiment" is the
#    permitted fallback when UC registration is unavailable).
import os
dbutils.fs.mkdirs(GENAI2_ROOT + "/evidence/screenshots")

payload = {
    "mode": "uc",
    "model": MODEL_NAME,
    "versions": [
        {"version": str(V1_VER), "agent_version": 1, "system_prompt_sha": V1_SHA},
        {"version": str(V2_VER), "agent_version": 2, "system_prompt_sha": V2_SHA, "alias": "champion"},
    ],
    "endpoint": ENDPOINT_NAME,
    "served_version": str(V2_VER),
    "screenshot": SCREENSHOT_REL,
    "runs": {"v1": V1_RUN, "v2": V2_RUN},
}
dbutils.fs.put(EVIDENCE_PATH, json.dumps(payload, indent=2), overwrite=True)
print("evidence written:", EVIDENCE_PATH)

In [0]:
# 6. Verify R7: re-read evidence from disk + fresh SHA recomputation + live registry + endpoint.
ev = json.loads(dbutils.fs.head(EVIDENCE_PATH, 200000))
assert ev["mode"] in ("uc", "experiment")
assert ev["model"] == MODEL_NAME and ev["endpoint"] == ENDPOINT_NAME

def sha_of(p):
    return hashlib.sha256(open(p).read().strip().encode()).hexdigest()[:12]

v1mv = client.get_model_version(MODEL_NAME, ev["versions"][0]["version"])
v2mv = client.get_model_version(MODEL_NAME, ev["versions"][1]["version"])
assert sha_of(V1_PATH) == ev["versions"][0]["system_prompt_sha"] == v1mv.tags["system_prompt_sha"]
assert sha_of(V2_PATH) == ev["versions"][1]["system_prompt_sha"] == v2mv.tags["system_prompt_sha"]

am = client.get_registered_model(MODEL_NAME).aliases
aliases = am if isinstance(am, dict) else {a.alias_name: a.version for a in (am or [])}
assert aliases.get("champion") == ev["versions"][1]["version"], aliases

exp = mlflow.get_experiment_by_name(EXPERIMENT)
r1 = client.search_runs([exp.experiment_id], filter_string="tags.agent_version = '1'")[0]
r2 = client.search_runs([exp.experiment_id], filter_string="tags.agent_version = '2'")[0]
assert r1.data.tags.get("system_prompt_sha") == sha_of(V1_PATH)
assert r2.data.tags.get("system_prompt_sha") == sha_of(V2_PATH) and r2.data.tags.get("alias") == "champion"

ep = w.serving_endpoints.get(ENDPOINT_NAME)
se = ep.config.served_entities[0]
assert se.entity_name == MODEL_NAME and se.entity_version == ev["versions"][1]["version"]
ready = ready_state(ep)
print(f"registry OK: v{ev['versions'][0]['version']} (v1 sha {ev['versions'][0]['system_prompt_sha']}) + "
      f"v{ev['versions'][1]['version']} (v2 sha {ev['versions'][1]['system_prompt_sha']}, alias champion)")
print(f"runs OK: agent_version 1/2 tagged with the right SHAs; v2 run alias=champion")
print(f"endpoint OK: {ENDPOINT_NAME} serves {MODEL_NAME} v{se.entity_version} | ready={ready}")

import os
shot = GENAI2_ROOT + "/" + SCREENSHOT_REL
if os.path.exists(shot):
    print("screenshot present:", SCREENSHOT_REL)
else:
    print(f"REMEMBER: drop the endpoint screenshot at {SCREENSHOT_REL} (serving UI -> claims-agent)")

assert ready == "READY", "endpoint not READY yet - re-run this cell in a few minutes"
print("ALL R7 SELF-CHECKS PASSED")